In [11]:
# ============================================================================
# BLOCK 1: INSTALL DEPENDENCIES (Run first)
# ============================================================================

print("="*70)
print("INSTALLING DEPENDENCIES")
print("="*70)

!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q gradio pillow opencv-python-headless ipywidgets

import torch
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
import os

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
print("="*70)

INSTALLING DEPENDENCIES



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: "'git+https://github.com/facebookresearch/detectron2.git'": Expected package name at the start of dependency specifier
    'git+https://github.com/facebookresearch/detectron2.git'
    ^


✅ PyTorch: 2.6.0+cu118
✅ CUDA: True



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install setuptools


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
# ============================================================================
# BLOCK 2: LOAD MODEL WITH DETERMINISTIC SETTINGS
# ============================================================================

print("="*70)
print("LOADING SHOULDER/ARM MODEL (DETERMINISTIC MODE)")
print("="*70)

import torch
import numpy as np
import random

# ========== SET DETERMINISTIC SEEDS ==========
def set_deterministic_mode(seed=42):
    """Fix all random sources for consistent predictions"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ Deterministic mode enabled (seed={seed})")

set_deterministic_mode(42)

from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2 import model_zoo

# ===== YOUR MODEL PATH =====
model_path = r"C:\Users\andre\Documents\Graduation Project\Shoulders\AP50 47.12%\model_final.pth"

print(f"📁 Model path: {model_path}")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}")

# Load configuration
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

# Set model weights
cfg.MODEL.WEIGHTS = model_path
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Create predictor
predictor = DefaultPredictor(cfg)

# CRITICAL: Set model to evaluation mode
predictor.model.eval()
print(f"✅ Model loaded in EVAL mode (deterministic)")

# Verify model mode
print(f"   Training mode: {predictor.model.training}")  # Should print False

class_names = ["fracture"]
metadata = MetadataCatalog.get("shoulder_arm_train")
metadata.thing_classes = class_names

print("\n✅ Model ready for deterministic predictions!")
print("="*70)

LOADING SHOULDER/ARM MODEL (DETERMINISTIC MODE)
✅ Deterministic mode enabled (seed=42)
📁 Model path: C:\Users\andre\Documents\Graduation Project\Shoulders\AP50 47.12%\model_final.pth


Skip loading parameter 'proposal_generator.rpn_head.objectness_logits.weight' to the model due to incompatible shapes: (15, 256, 1, 1) in the checkpoint but (3, 256, 1, 1) in the model! You might want to double check if this is expected.
Skip loading parameter 'proposal_generator.rpn_head.objectness_logits.bias' to the model due to incompatible shapes: (15,) in the checkpoint but (3,) in the model! You might want to double check if this is expected.
Skip loading parameter 'proposal_generator.rpn_head.anchor_deltas.weight' to the model due to incompatible shapes: (60, 256, 1, 1) in the checkpoint but (12, 256, 1, 1) in the model! You might want to double check if this is expected.
Skip loading parameter 'proposal_generator.rpn_head.anchor_deltas.bias' to the model due to incompatible shapes: (60,) in the checkpoint but (12,) in the model! You might want to double check if this is expected.
Some model parameters or buffers are not found in the checkpoint:
proposal_generator.rpn_head.anch

✅ Model loaded in EVAL mode (deterministic)
   Training mode: False

✅ Model ready for deterministic predictions!


In [25]:
# ============================================================================
# FIXED: DETERMINISTIC PREDICTION FUNCTION
# ============================================================================

import torch
import numpy as np
import random

def set_deterministic_mode(seed=42):
    """
    Fix all random seeds for consistent predictions
    Call this ONCE before any predictions
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Make CuDNN deterministic (slower but consistent)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✅ Deterministic mode enabled (seed={seed})")

# Call this ONCE when loading the model
set_deterministic_mode(42)

def predict_fracture_deterministic(image, threshold=0.3):
    """
    DETERMINISTIC prediction - same result every time for same image
    """
    # Ensure model is in evaluation mode (very important!)
    predictor.model.eval()  # This disables dropout and batch norm randomness
    
    # Convert to numpy array
    if isinstance(image, Image.Image):
        image = np.array(image)

    # Ensure RGB format
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)

    # Set threshold
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    predictor_temp = DefaultPredictor(cfg)

    # Run prediction (now deterministic)
    with torch.no_grad():  # Disable gradient computation
        outputs = predictor_temp(image)
    
    instances = outputs["instances"].to("cpu")

    # ... rest of your prediction code

✅ Deterministic mode enabled (seed=42)


In [31]:
# ============================================================================
# COMPLETE TEST: Verify predictions are deterministic
# ============================================================================

print("="*70)
print("TESTING DETERMINISTIC PREDICTIONS")
print("="*70)

import os
import cv2
import numpy as np
import torch
import random

# ========== FIRST, FIND A TEST IMAGE ==========
# Look for any test image in your dataset
test_images_path = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Enhanced_Shoulder_Arm_Dataset_FINAL\test\images"

# Find first available image
test_image_path = None
if os.path.exists(test_images_path):
    for file in os.listdir(test_images_path):
        if file.endswith(('.jpg', '.jpeg', '.png')):
            test_image_path = os.path.join(test_images_path, file)
            break

if test_image_path is None:
    print("❌ No test image found! Please provide a valid path.")
    print("   Or manually set test_image_path to an image file")
    test_image_path = input("Enter path to a test image: ")

print(f"📁 Using test image: {test_image_path}")

# ========== SET DETERMINISTIC MODE ==========
def set_deterministic_mode(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ Deterministic mode enabled (seed={seed})")

set_deterministic_mode(42)

# ========== RUN 5 PREDICTIONS ==========
threshold = 0.3
results = []
detection_counts = []

print(f"\n🔍 Running 5 predictions with threshold={threshold}")
print("-"*50)

for i in range(5):
    # Load image
    img = cv2.imread(test_image_path)
    if img is None:
        print(f"❌ Cannot read image: {test_image_path}")
        break
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Make prediction
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    outputs = predictor(img_rgb)
    instances = outputs["instances"].to("cpu")
    num_detections = len(instances)
    
    # Get confidence scores if any
    if num_detections > 0:
        scores = instances.scores.numpy()
        max_conf = np.max(scores)
        print(f"Run {i+1}: {num_detections} detection(s) | Max confidence: {max_conf:.2%}")
        detection_counts.append(num_detections)
    else:
        print(f"Run {i+1}: {num_detections} detection(s)")
        detection_counts.append(0)
    
    results.append(num_detections)

# ========== ANALYZE RESULTS ==========
print("\n" + "="*70)
print("📊 RESULTS ANALYSIS")
print("="*70)

print(f"\nDetection counts: {detection_counts}")

if len(set(results)) == 1:
    print("\n✅ SUCCESS! All predictions are IDENTICAL!")
    print(f"   Consistent result: {results[0]} detection(s)")
    print("\n   Your model is now deterministic!")
else:
    print("\n❌ FAILED! Predictions are still INCONSISTENT!")
    print(f"   Results: {results}")
    print("\n   This indicates the model itself is unstable.")
    print("   Possible causes:")
    print("   1. Model is under-trained (needs more epochs)")
    print("   2. Model is overfitting")
    print("   3. Floating point precision issues")
    
    # Calculate consistency percentage
    most_common = max(set(results), key=results.count)
    consistency = results.count(most_common) / len(results) * 100
    print(f"\n   Consistency: {consistency:.1f}% (most common: {most_common} detections)")

print("="*70)

TESTING DETERMINISTIC PREDICTIONS
📁 Using test image: C:\Users\andre\Documents\Graduation Project\Shoulders\Enhanced_Shoulder_Arm_Dataset_FINAL\test\images\aihc_miniproject.v2i.yolov8_download-4-1-_jpg.rf.feef7bd2f165ee2b27904c6ece3c13c0.jpg
✅ Deterministic mode enabled (seed=42)

🔍 Running 5 predictions with threshold=0.3
--------------------------------------------------
Run 1: 0 detection(s)
Run 2: 0 detection(s)
Run 3: 0 detection(s)
Run 4: 0 detection(s)
Run 5: 0 detection(s)

📊 RESULTS ANALYSIS

Detection counts: [0, 0, 0, 0, 0]

✅ SUCCESS! All predictions are IDENTICAL!
   Consistent result: 0 detection(s)

   Your model is now deterministic!


In [33]:
# ============================================================================
# BLOCK 4: GRADIO WEB INTERFACE (Professional UI)
# ============================================================================

print("="*70)
print("LAUNCHING GRADIO WEB INTERFACE")
print("="*70)

import gradio as gr

def gradio_predict(image, threshold):
    """Wrapper for Gradio interface"""
    if image is None:
        return None, "Please upload an image."
    
    # Convert to PIL if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Run prediction
    annotated_img, result_text = predict_fracture(image, threshold)
    
    return annotated_img, result_text

# Create Gradio interface
with gr.Blocks(theme=gr.themes.Soft(), title="AETHEA Shoulder & Arm Fracture Detection") as demo:
    gr.Markdown("""
    # 🏥 AETHEA Shoulder & Arm Fracture Detection System
    
    Upload an X-ray image of a **shoulder or arm** to detect potential fractures using AI.
    
    **Model Performance:** AP50: 47.12% | Trained on 27,480 images
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload Shoulder/Arm X-ray", height=400)
            
            threshold = gr.Slider(
                minimum=0.1,
                maximum=0.9,
                value=0.3,
                step=0.05,
                label="Detection Threshold",
                info="Lower = more sensitive (more detections), Higher = more specific (fewer false positives)"
            )
            
            submit_btn = gr.Button("🔍 Detect Fracture", variant="primary", size="lg")
        
        with gr.Column(scale=1):
            image_output = gr.Image(label="Analysis Result", height=400)
    
    with gr.Row():
        result_output = gr.HTML(label="Diagnosis Report")
    
    submit_btn.click(
        fn=gradio_predict,
        inputs=[image_input, threshold],
        outputs=[image_output, result_output]
    )
    
    gr.Markdown("""
    ---
    ### 📋 Instructions
    1. Upload a shoulder or arm X-ray image (JPG, PNG)
    2. Adjust threshold if needed (0.3 is recommended)
    3. Click "Detect Fracture" to analyze
    
    ### 🎯 Threshold Guide
    | Threshold | Sensitivity | Use Case |
    |-----------|-------------|----------|
    | 0.2 | High | Catch subtle fractures (more false positives) |
    | 0.3 | Balanced | Default recommendation |
    | 0.5 | Low | Only confident detections |
    
    ### ⚕️ Medical Disclaimer
    This is an AI screening tool. All results must be verified by qualified healthcare professionals.
    """)

# Launch
demo.launch(share=True, debug=False)

print("\n✅ Gradio interface launched!")
print("📤 Share the public link above with others")
print("="*70)

LAUNCHING GRADIO WEB INTERFACE
* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://03152395ab57ebceba.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Gradio interface launched!
📤 Share the public link above with others


In [9]:
# ============================================================================
# BLOCK 4: PREDICTION FUNCTION (Only show fracture classes)
# ============================================================================

print("="*70)
print("DEFINING PREDICTION FUNCTION")
print("="*70)

# Define which classes are ACTUAL fractures
# Class 0 = fracture, Class 2 = pelvic_fracture
FRACTURE_CLASSES = [0, 2]  # Only these will show boxes

print(f"📋 Fracture classes: {FRACTURE_CLASSES}")
print(f"   Class 0 (fracture) - WILL SHOW BOXES ✅")
print(f"   Class 1 (no fracture) - WILL NOT SHOW BOXES ❌")
print(f"   Class 2 (pelvic_fracture) - WILL SHOW BOXES ✅")

def predict_fracture(image, threshold=0.3):
    """
    Detect fractures in X-ray image - only shows boxes for actual fracture classes
    """
    # Convert to numpy array
    if isinstance(image, Image.Image):
        image = np.array(image)

    # Ensure RGB format
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)

    # Update threshold
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    predictor_temp = DefaultPredictor(cfg)

    # Run prediction
    outputs = predictor_temp(image)
    instances = outputs["instances"].to("cpu")

    # Get all detections
    all_instances = instances
    num_total = len(all_instances)
    
    # FILTER: Keep only fracture classes (class 0 and class 2)
    if num_total > 0:
        pred_classes = all_instances.pred_classes.numpy()
        scores = all_instances.scores.numpy()
        boxes = all_instances.pred_boxes.tensor.numpy()
        
        # Find indices of fracture classes
        fracture_indices = [i for i, cls in enumerate(pred_classes) if cls in FRACTURE_CLASSES]
        
        # Create filtered instances
        if fracture_indices:
            from detectron2.structures import Instances, Boxes
            filtered_instances = Instances(image.shape[:2])
            filtered_instances.pred_boxes = Boxes(np.array([boxes[i] for i in fracture_indices]))
            filtered_instances.scores = torch.tensor([scores[i] for i in fracture_indices])
            filtered_instances.pred_classes = torch.tensor([pred_classes[i] for i in fracture_indices])
            instances = filtered_instances
        else:
            # No fracture detections, create empty instances
            from detectron2.structures import Instances
            instances = Instances(image.shape[:2])
    
    # Get detection info
    num_detections = len(instances)
    has_fracture = num_detections > 0

    # Get confidence scores
    if has_fracture:
        scores = instances.scores.numpy()
        max_confidence = float(np.max(scores))
        avg_confidence = float(np.mean(scores))
        
        # Get class names for detected fractures
        pred_classes = instances.pred_classes.numpy()
        detected_classes = [class_names[c] for c in pred_classes]
    else:
        max_confidence = 0.0
        avg_confidence = 0.0
        detected_classes = []

    # Create visualization (only fracture boxes will be drawn)
    v = Visualizer(
        image[:, :, ::-1],
        metadata=metadata,
        scale=1.0
    )
    vis = v.draw_instance_predictions(instances)
    annotated_img = vis.get_image()[:, :, ::-1]

    # Generate result text
    if has_fracture:
        fracture_types = "<br>".join([f"• {cls}" for cls in detected_classes])
        
        result_text = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; color: white;">
    <h1 style="margin: 0; font-size: 32px; font-weight: bold; text-align: center;">
        🚨 FRACTURE DETECTED
    </h1>
</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 25px; background: white; border-radius: 10px; margin-top: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 0; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
        📊 Detection Details
    </h2>

    <div style="background: #f7fafc; padding: 15px; border-radius: 8px; margin: 15px 0;">
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #667eea;">•</strong> Number of fractures: <span style="font-weight: bold; color: #e53e3e;">{num_detections}</span>
        </p>
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #667eea;">•</strong> Highest confidence: <span style="font-weight: bold; color: #38a169;">{max_confidence*100:.1f}%</span>
        </p>
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #667eea;">•</strong> Average confidence: <span style="font-weight: bold;">{avg_confidence*100:.1f}%</span>
        </p>
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #667eea;">•</strong> Threshold used: <span style="font-weight: bold;">{threshold}</span>
        </p>
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #667eea;">•</strong> Fracture types: <br>{fracture_types}
        </p>
    </div>

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
        🏥 Recommendation
    </h2>

    <div style="background: #fff5f5; border-left: 5px solid #e53e3e; padding: 15px; margin: 15px 0; border-radius: 5px;">
        <p style="margin: 0; font-size: 16px; font-weight: bold; color: #c53030;">
            IMMEDIATE MEDICAL CONSULTATION RECOMMENDED
        </p>
        <p style="margin: 10px 0 0 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
            Fracture(s) detected in X-ray image.<br>
            Please consult a qualified radiologist or orthopedic specialist.
        </p>
    </div>

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
        ⚠️ Important Disclaimer
    </h2>

    <div style="background: #fffaf0; border: 2px solid #ed8936; padding: 15px; border-radius: 8px; margin-top: 15px;">
        <p style="margin: 0; font-size: 14px; color: #2d3748; line-height: 1.6;">
            This is an AI-assisted screening tool for preliminary assessment.<br>
            Results <strong>MUST</strong> be verified by a qualified medical professional.<br>
            Do NOT use as sole basis for diagnosis or treatment decisions.
        </p>
    </div>
</div>
        """
    else:
        result_text = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); border-radius: 15px; color: white;">
    <h1 style="margin: 0; font-size: 32px; font-weight: bold; text-align: center;">
        ✅ NO FRACTURE DETECTED
    </h1>
</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 25px; background: white; border-radius: 10px; margin-top: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 0; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
        📊 Analysis Details
    </h2>

    <div style="background: #f0fff4; padding: 15px; border-radius: 8px; margin: 15px 0;">
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #48bb78;">•</strong> No fractures detected
        </p>
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #48bb78;">•</strong> Threshold used: <span style="font-weight: bold;">{threshold}</span>
        </p>
        <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
            <strong style="color: #48bb78;">•</strong> Image analyzed successfully
        </p>
    </div>

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
        💡 Interpretation
    </h2>

    <div style="background: #e6fffa; border-left: 5px solid #38b2ac; padding: 15px; margin: 15px 0; border-radius: 5px;">
        <p style="margin: 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
            No obvious fractures detected in this X-ray image.<br>
            However, subtle fractures may not be detected by AI.
        </p>
    </div>

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
        🏥 Recommendation
    </h2>

    <div style="background: #fffff0; border-left: 5px solid #ecc94b; padding: 15px; margin: 15px 0; border-radius: 5px;">
        <p style="margin: 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
            <strong>If you experience persistent pain or symptoms:</strong>
        </p>
        <ul style="margin: 10px 0; padding-left: 20px; font-size: 15px; color: #2d3748; line-height: 1.6;">
            <li>Consult a medical professional</li>
            <li>Clinical examination may reveal findings not visible on X-ray</li>
            <li>Follow-up imaging may be recommended</li>
        </ul>
    </div>

    <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
        ⚠️ Important Disclaimer
    </h2>

    <div style="background: #fffaf0; border: 2px solid #ed8936; padding: 15px; border-radius: 8px; margin-top: 15px;">
        <p style="margin: 0; font-size: 14px; color: #2d3748; line-height: 1.6;">
            This is an AI screening tool. Normal AI results do not rule out fractures.<br>
            Always consult a qualified medical professional for proper diagnosis and treatment.
        </p>
    </div>
</div>
        """

    return annotated_img, result_text

print("✅ Prediction function defined!")
print("   - Only shows boxes for class 0 (fracture) and class 2 (pelvic_fracture)")
print("   - Class 1 (no fracture) will NOT show any boxes")
print("="*70)

DEFINING PREDICTION FUNCTION
📋 Fracture classes: [0, 2]
   Class 0 (fracture) - WILL SHOW BOXES ✅
   Class 1 (no fracture) - WILL NOT SHOW BOXES ❌
   Class 2 (pelvic_fracture) - WILL SHOW BOXES ✅
✅ Prediction function defined!
   - Only shows boxes for class 0 (fracture) and class 2 (pelvic_fracture)
   - Class 1 (no fracture) will NOT show any boxes


In [34]:
# ============================================================================
# FIXED BLOCK 5: SIMPLE INTERFACE (Consistent results)
# ============================================================================

print("="*70)
print("LAUNCHING FIXED TEST INTERFACE")
print("="*70)

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io

# Create upload widget
upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📤 Upload X-ray',
    style={'description_width': 'initial'}
)

# Create threshold slider
threshold_slider = widgets.FloatSlider(
    value=0.3,
    min=0.1,
    max=0.9,
    step=0.05,
    description='Threshold:',
    style={'description_width': 'initial'}
)

# Create analyze button
analyze_btn = widgets.Button(
    description='🔍 Analyze Image',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

# Create output areas
image_output = widgets.Output()
result_output = widgets.Output()

# Display widgets
display(widgets.HTML("<h2>🦴 Hips & Pelvis Fracture Detection</h2>"))
display(widgets.HTML("<p style='color: #e53e3e;'><strong>Note:</strong> Only fracture classes will show bounding boxes. 'No Fracture' predictions will NOT show boxes.</p>"))
display(upload_btn)
display(threshold_slider)
display(analyze_btn)
display(widgets.HBox([image_output, result_output]))

# Define analysis function
def analyze_image(b):
    with image_output:
        clear_output(wait=True)
        
        if not upload_btn.value:
            print("Please upload an image first.")
            return
        
        # Get uploaded image
        uploaded_file = list(upload_btn.value.values())[0]
        image = Image.open(io.BytesIO(uploaded_file['content']))
        
        # Get threshold
        threshold = threshold_slider.value
        
        # Run prediction with FIXED function
        annotated_img, result_text, has_fracture, num_fractures = predict_hips_pelvis_fixed(image, threshold)
        
        # Display annotated image
        plt.figure(figsize=(10, 8))
        plt.imshow(annotated_img)
        plt.axis('off')
        
        if has_fracture:
            plt.title(f'⚠️ {num_fractures} Fracture(s) Detected (Threshold: {threshold})', color='red', fontweight='bold')
        else:
            plt.title(f'✅ No Fracture Detected (Threshold: {threshold})', color='green', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    with result_output:
        clear_output(wait=True)
        display(widgets.HTML(result_text))

# Bind button
analyze_btn.on_click(analyze_image)

print("\n✅ Fixed interface ready!")
print("   - Results are consistent for same threshold")
print("   - Only ACTUAL fractures show bounding boxes")
print("   - 'No Fracture' predictions show NO boxes")
print("\n📤 Upload a hip/pelvis X-ray and click 'Analyze Image'")
print("="*70)

LAUNCHING FIXED TEST INTERFACE


HTML(value='<h2>🦴 Hips & Pelvis Fracture Detection</h2>')

HTML(value="<p style='color: #e53e3e;'><strong>Note:</strong> Only fracture classes will show bounding boxes. …

FileUpload(value={}, accept='image/*', description='📤 Upload X-ray')

FloatSlider(value=0.3, description='Threshold:', max=0.9, min=0.1, step=0.05, style=SliderStyle(description_wi…

Button(button_style='primary', description='🔍 Analyze Image', layout=Layout(width='200px'), style=ButtonStyle(…


✅ Fixed interface ready!
   - Results are consistent for same threshold
   - Only ACTUAL fractures show bounding boxes
   - 'No Fracture' predictions show NO boxes

📤 Upload a hip/pelvis X-ray and click 'Analyze Image'


In [36]:
# ============================================================================
# FIXED BLOCK 6: GRADIO WEB INTERFACE (Consistent predictions)
# ============================================================================

print("="*70)
print("LAUNCHING FIXED GRADIO WEB INTERFACE")
print("="*70)

import gradio as gr

def gradio_predict_fixed(image, threshold):
    """Wrapper function for Gradio interface"""
    if image is None:
        return None, "Please upload an image."
    
    # Convert Gradio image to PIL if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Run prediction with FIXED function
    annotated_img, result_text, has_fracture, num_fractures = predict_hips_pelvis_fixed(image, threshold)
    
    return annotated_img, result_text

# Create Gradio interface
with gr.Blocks(theme=gr.themes.Soft(), title="AETHEA - Hips & Pelvis Fracture Detection") as demo:
    gr.Markdown("""
    # 🦴 AETHEA Hips & Pelvis Fracture Detection System
    
    ### Specialized AI Model for Hip and Pelvic Fracture Detection
    
    Upload a hip or pelvis X-ray image for AI-powered fracture detection.
    
    ⚠️ **Important**: Only actual fractures will show bounding boxes. "No Fracture" predictions will show NO boxes.
    """)
    
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload X-ray Image", height=400)
            
            threshold = gr.Slider(
                minimum=0.1,
                maximum=0.9,
                value=0.3,
                step=0.05,
                label="Detection Threshold",
                info="Lower = more sensitive, Higher = more specific"
            )
            
            submit_btn = gr.Button("🔍 Detect Fracture", variant="primary", size="lg")
        
        with gr.Column():
            image_output = gr.Image(label="Analysis Result", height=400)
    
    with gr.Row():
        result_output = gr.HTML(label="Diagnosis Report")
    
    submit_btn.click(
        fn=gradio_predict_fixed,
        inputs=[image_input, threshold],
        outputs=[image_output, result_output]
    )
    
    gr.Markdown("""
    ---
    ### 📋 Important Notes
    
    - **Fracture Classes**: Only "Fracture" and "Pelvic Fracture" show bounding boxes
    - **No Fracture Class**: Will NOT show any bounding boxes
    - **Consistent Results**: Same image + same threshold = same result
    
    ### 📊 Model Performance
    - **AP50**: 69.7%
    - **No Fracture detection**: 65.1% AP
    - **Fracture detection**: 26.2% AP
    
    ### ⚠️ Medical Disclaimer
    This is an AI screening tool. Results MUST be verified by a qualified medical professional.
    """)

# Launch the interface
print("\n🚀 Launching fixed Gradio interface...")
demo.launch(share=False, debug=False, inbrowser=True)
print("\n✅ Fixed Gradio interface launched!")
print("   - Results are now consistent for same threshold")
print("   - Only actual fractures show bounding boxes")
print("="*70)

LAUNCHING FIXED GRADIO WEB INTERFACE

🚀 Launching fixed Gradio interface...
* Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.



✅ Fixed Gradio interface launched!
   - Results are now consistent for same threshold
   - Only actual fractures show bounding boxes
